In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [8]:
run='oper_01'

In [ ]:
path = f'/jetfs/scratch/psplechtna/flexpart_output/oper/2024/{run}/processed_data/'
levels = ['50.0', '100.0', '200.0', '500.0', '1000.0', '2000.0', 'bl']
lh = xr.concat(
    [xr.open_dataset(f'{path}lh_{lev}.zarr').lh_bl for lev in levels],
    dim=pd.Index(levels, name='level'),
)

sh = xr.concat(
    [xr.open_dataset(f'{path}sh_{lev}.zarr').sh_bl for lev in levels],
    dim=pd.Index(levels, name='level'),
)

In [11]:
tsr = xr.open_dataset(f'/jetfs/scratch/psplechtna/thesisdata/oper/2024/{run}_tsr.nc')

In [12]:
tsr

<xarray.Dataset> Size: 2kB
Dimensions:       (release_date: 24)
Coordinates:
  * release_date  (release_date) datetime64[ns] 192B 2024-06-01 ... 2024-06-1...
Data variables:
    diag          (release_date) float32 96B ...
    diag_anomaly  (release_date) float32 96B ...
    init          (release_date) float64 192B ...
    init_anomaly  (release_date) float64 192B ...
    flux_lh       (release_date) float32 96B ...
    flux_sh       (release_date) float32 96B ...
    sw_rad        (release_date) float32 96B ...
    lw_rad        (release_date) float32 96B ...
    advection     (release_date) float32 96B ...
    seasonality   (release_date) float32 96B ...
    calc          (release_date) float64 192B ...
    calc_wo       (release_date) float64 192B ...

In [3]:
sum_lh = lh.sum(dim=['latitude', 'longitude']).isel(time=slice(None, None, -1)).load()
sum_sh = sh.sum(dim=['latitude', 'longitude']).isel(time=slice(None, None, -1)).load()

In [16]:
ds = xr.merge([
    sum_lh.sum(dim='time').rename('lh_per_rel'),
    sum_sh.sum(dim='time').rename('sh_per_rel'),
])

/tmp/ipykernel_3047800/1766052857.py:1: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds = xr.merge([


In [17]:
ds = ds.assign_coords(release_time=('pointspec', tsr['release_date'].values))

In [19]:
ds.to_netcdf(f'/jetfs/scratch/psplechtna/thesisdata/oper/2024/heighttest_{run}.nc')